[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/polars-certified/notebooks/day-10-capstone-production-analytics-pipeline.ipynb#scrollTo=f1a2b3c4)

---
# Day 10 · Capstone: Production Analytics Pipeline
**certified-journeys / polars-certified** &nbsp;|&nbsp; Capstone

> **Goal for today:** Build a complete, production-quality Polars analytics pipeline combining all concepts from Days 1–9.

---
## What you will build

A full e-commerce analytics pipeline across three related tables:

```
customers (50k rows)  ─┬─────────┐
products  (50k rows)  ─┼─ orders ─┤  →  clean →  enrich →  aggregate →  export
orders    (50k rows)  ─┴─────────┘
```

Pipeline stages:
1. Generate synthetic data
2. Ingest with schema validation (lazy)
3. Clean: nulls, types, deduplication
4. Feature engineering: derived columns (expressions only, no UDFs)
5. Multi-table join
6. Window analytics: revenue rank, 7-day rolling revenue, MoM growth
7. Aggregation: monthly summary by category and region
8. Export: partitioned Parquet + summary CSV
9. Unit tests with pytest
10. Production checklist + portfolio README

In [ ]:
%pip install -q polars pyarrow pytest

---
## Step 1 · Generate a realistic e-commerce dataset

Three related tables with referential integrity, realistic distributions,
and intentional data quality issues for the cleaning step to catch.

In [ ]:
import polars as pl
import numpy as np
import datetime, os, pathlib, tempfile

rng = np.random.default_rng(2024)
N = 50_000  # rows per table

CATEGORIES = ["Electronics", "Clothing", "Books", "Home", "Sports"]
REGIONS    = ["US", "EU", "APAC", "LATAM"]
STATUSES   = ["completed", "pending", "refunded", "cancelled"]

# --- customers ---
customers_raw = pl.DataFrame({
    "customer_id": np.arange(1, N + 1),
    "name":        [f"Customer_{i}" for i in range(1, N + 1)],
    "region":      rng.choice(REGIONS, N).tolist(),
    "signup_date": [
        (datetime.date(2020, 1, 1) + datetime.timedelta(days=int(d))).isoformat()
        for d in rng.integers(0, 1460, N)
    ],
    "is_premium":  rng.choice([True, False, None], N, p=[0.3, 0.65, 0.05]).tolist(),
})

# --- products ---
products_raw = pl.DataFrame({
    "product_id":  np.arange(1, N + 1),
    "product_name": [f"Product_{i}" for i in range(1, N + 1)],
    "category":    rng.choice(CATEGORIES, N).tolist(),
    "unit_price":  rng.uniform(5.0, 500.0, N).round(2),
    "stock":       rng.integers(0, 1000, N).tolist(),
})

# --- orders (with intentional data quality issues) ---
order_dates = [
    (datetime.date(2023, 1, 1) + datetime.timedelta(days=int(d))).isoformat()
    for d in rng.integers(0, 730, N)
]
orders_raw = pl.DataFrame({
    "order_id":    np.arange(1, N + 1),
    "customer_id": rng.integers(1, N + 1, N).tolist(),  # some may be invalid
    "product_id":  rng.integers(1, N + 1, N).tolist(),
    "quantity":    rng.integers(1, 10, N).tolist(),
    "discount":    rng.choice([0.0, 0.05, 0.10, 0.15, None], N,
                               p=[0.5, 0.2, 0.15, 0.1, 0.05]).tolist(),
    "status":      rng.choice(STATUSES, N).tolist(),
    "order_date":  order_dates,
})

# Inject duplicates (3% of orders)
dup_idx = rng.integers(0, N, int(N * 0.03))
orders_with_dups = pl.concat([orders_raw, orders_raw[dup_idx]], rechunk=True)

TMPDIR = tempfile.mkdtemp()

# Write to CSV (simulating raw ingestion files)
customers_raw.write_csv(os.path.join(TMPDIR, "customers.csv"))
products_raw.write_csv(os.path.join(TMPDIR, "products.csv"))
orders_with_dups.write_csv(os.path.join(TMPDIR, "orders.csv"))

print(f"customers: {customers_raw.shape}")
print(f"products:  {products_raw.shape}")
print(f"orders (with dups): {orders_with_dups.shape}")
print(f"\nFiles written to: {TMPDIR}")

**What just happened?**
- Three related DataFrames with ~50k rows each, written to CSV to simulate a real ingestion scenario.
- **Intentional data quality issues**: 5% null `is_premium`, 5% null `discount`, 3% duplicate orders.
- The cleaning step (Step 3) will handle all of these systematically.
- All data is generated synthetically — no external downloads needed, reproducible with the fixed seed.

---
## Step 2 · Lazy ingestion with schema validation

In production, never load raw files eagerly. Always validate schema at scan time
so bad files are caught before any compute runs.

In [ ]:
# Define expected schemas explicitly — don't trust auto-inference for production
CUSTOMER_SCHEMA = {
    "customer_id": pl.Int64,
    "name":        pl.String,
    "region":      pl.String,
    "signup_date": pl.String,   # will cast to Date in cleaning step
    "is_premium":  pl.Boolean,
}

PRODUCT_SCHEMA = {
    "product_id":   pl.Int64,
    "product_name": pl.String,
    "category":     pl.String,
    "unit_price":   pl.Float64,
    "stock":        pl.Int64,
}

ORDER_SCHEMA = {
    "order_id":    pl.Int64,
    "customer_id": pl.Int64,
    "product_id":  pl.Int64,
    "quantity":    pl.Int64,
    "discount":    pl.Float64,
    "status":      pl.String,
    "order_date":  pl.String,
}

def ingest_lazy(path: str, schema: dict) -> pl.LazyFrame:
    """Scan a CSV lazily, validating column names against expected schema."""
    lf = pl.scan_csv(path, schema_overrides=schema)
    # Validate: all expected columns must exist
    actual_cols = set(lf.collect_schema().names())
    expected_cols = set(schema.keys())
    missing = expected_cols - actual_cols
    if missing:
        raise ValueError(f"Schema validation failed: missing columns {missing}")
    return lf

# Ingest all three tables lazily
lf_customers = ingest_lazy(os.path.join(TMPDIR, "customers.csv"), CUSTOMER_SCHEMA)
lf_products  = ingest_lazy(os.path.join(TMPDIR, "products.csv"),  PRODUCT_SCHEMA)
lf_orders    = ingest_lazy(os.path.join(TMPDIR, "orders.csv"),    ORDER_SCHEMA)

print("All tables ingested lazily. Schema validated.")
print("customers schema:", lf_customers.collect_schema())
print("\norders schema:",   lf_orders.collect_schema())

**What just happened?**
- `schema_overrides` enforces column types at scan time — type mismatches raise errors before any row is processed.
- `collect_schema()` inspects the schema **without reading data** — it's instant even for huge files.
- The `ingest_lazy` function is a reusable pattern: wrap every raw ingestion with schema validation.
- **No data has been loaded into memory yet** — all three `lf_*` variables are query plans.

---
## Step 3 · Data cleaning: nulls, types, deduplication

Every cleaning operation is a pure Polars expression — no UDFs.
The cleaning functions are kept separate so they can be unit-tested independently.

In [ ]:
def clean_customers(lf: pl.LazyFrame) -> pl.LazyFrame:
    return (
        lf
        # Cast signup_date from String to Date
        .with_columns(pl.col("signup_date").str.to_date("%Y-%m-%d"))
        # Fill null is_premium with False (conservative default)
        .with_columns(pl.col("is_premium").fill_null(False))
        # Cast region to Categorical for memory efficiency
        .with_columns(pl.col("region").cast(pl.Categorical))
    )

def clean_products(lf: pl.LazyFrame) -> pl.LazyFrame:
    return (
        lf
        # Drop products with zero price (data quality issue)
        .filter(pl.col("unit_price") > 0)
        # Cast category to Categorical
        .with_columns(pl.col("category").cast(pl.Categorical))
    )

def clean_orders(lf: pl.LazyFrame) -> pl.LazyFrame:
    return (
        lf
        # Remove duplicate orders (keep first occurrence by order_id)
        .unique(subset=["order_id"], keep="first")
        # Cast order_date
        .with_columns(pl.col("order_date").str.to_date("%Y-%m-%d"))
        # Fill null discount with 0.0 (no discount)
        .with_columns(pl.col("discount").fill_null(0.0))
        # Remove cancelled orders from the analytics pipeline
        .filter(pl.col("status") != "cancelled")
        # Cast status to Categorical
        .with_columns(pl.col("status").cast(pl.Categorical))
    )

# Apply cleaning
lf_customers_clean = clean_customers(lf_customers)
lf_products_clean  = clean_products(lf_products)
lf_orders_clean    = clean_orders(lf_orders)

# Collect customers to check null handling
cust_sample = lf_customers_clean.collect()
print(f"Customers after cleaning: {cust_sample.shape}")
print(f"Null is_premium: {cust_sample['is_premium'].null_count()}")

# Check order deduplication
orders_clean = lf_orders_clean.collect()
print(f"\nOrders after dedup + cancel filter: {orders_clean.shape[0]:,}")
print(f"Null discount: {orders_clean['discount'].null_count()}")

**What just happened?**
- Each cleaning function takes a `LazyFrame` and returns a `LazyFrame` — **composable and testable**.
- `unique(subset=["order_id"], keep="first")` removes duplicates in a single lazy operation.
- `fill_null(0.0)` and `fill_null(False)` are documented business decisions: null means no discount / not premium.
- **Separating cleaning functions** from the pipeline orchestration is the single most important structural decision — it's what makes the code testable.

---
## Step 4 · Feature engineering with pure expressions

All derived columns are computed as Polars expressions — no UDFs.
This keeps them optimizer-visible, parallel, and fast.

In [ ]:
def engineer_order_features(lf: pl.LazyFrame) -> pl.LazyFrame:
    return lf.with_columns([
        # Year and month for time-series aggregations
        pl.col("order_date").dt.year().alias("year"),
        pl.col("order_date").dt.month().alias("month"),
        # Year-month string for partitioning
        pl.col("order_date").dt.strftime("%Y-%m").alias("year_month"),
        # Day of week (0=Monday) — useful for seasonality analysis
        pl.col("order_date").dt.weekday().alias("day_of_week"),
        # Gross amount before discount
        # Note: unit_price will be joined later; here we just mark the formula
        # After join: (quantity * unit_price * (1 - discount)).alias("net_amount")
    ])

lf_orders_featured = engineer_order_features(lf_orders_clean)
sample = lf_orders_featured.collect()
print("Featured orders schema:")
print(sample.schema)
print(sample.select(["order_id", "order_date", "year", "month", "day_of_week"]).head(5))

**What just happened?**
- `dt.year()`, `dt.month()`, `dt.weekday()`, `dt.strftime()` are all built-in Polars date expressions — no UDFs.
- These run in parallel across the column using Rust SIMD — extracting 4 date fields from 50k rows takes microseconds.
- The `year_month` string column will be used for **partition naming** in the export step.
- **Net amount** calculation is deferred to after the join, where `unit_price` is available.

---
## Step 5 · Multi-table join pipeline

Join all three tables into a single enriched orders DataFrame.
Use `inner` joins for referential integrity — orphaned orders are dropped here deliberately.

In [ ]:
def build_enriched_orders(
    lf_orders: pl.LazyFrame,
    lf_customers: pl.LazyFrame,
    lf_products: pl.LazyFrame,
) -> pl.LazyFrame:
    """Join orders with customers and products, compute net_amount."""
    return (
        lf_orders
        # Join customers: add region, is_premium
        .join(
            lf_customers.select(["customer_id", "region", "is_premium"]),
            on="customer_id",
            how="inner",  # drop orders with no matching customer
        )
        # Join products: add category, unit_price
        .join(
            lf_products.select(["product_id", "category", "unit_price"]),
            on="product_id",
            how="inner",  # drop orders with no matching product
        )
        # Compute net amount after discount
        .with_columns(
            (pl.col("quantity") * pl.col("unit_price") * (1 - pl.col("discount")))
            .round(2)
            .alias("net_amount")
        )
    )

lf_enriched = build_enriched_orders(
    lf_orders_featured,
    lf_customers_clean,
    lf_products_clean,
)

enriched = lf_enriched.collect()
print(f"Enriched orders: {enriched.shape}")
print(enriched.select(["order_id", "category", "region", "net_amount",
                        "year_month", "is_premium"]).head(5))

**What just happened?**
- Three lazy DataFrames were joined in a single pipeline — Polars optimizes the join order and applies projection pushdown.
- Using `.select()` before the join limits which columns are brought in — this is **explicit projection pruning** that speeds up the join.
- `net_amount` is computed **after** the join, where both `quantity`, `unit_price`, and `discount` are available in one row.
- The `inner` join strategy is a documented business decision: orders with missing customer/product data are excluded from analytics.

---
## Step 6 · Window function analytics

Three analytical patterns: revenue rank per category, 7-day rolling revenue, and month-over-month growth.

In [ ]:
# First build daily revenue per category (needed for rolling + MoM)
daily_cat_revenue = (
    enriched
    .group_by(["order_date", "category"])
    .agg(pl.col("net_amount").sum().alias("daily_revenue"))
    .sort(["category", "order_date"])
)
print("Daily category revenue (sample):")
print(daily_cat_revenue.head(8))

# 1. Revenue rank within each category per day
ranked = daily_cat_revenue.with_columns(
    pl.col("daily_revenue")
      .rank(method="dense", descending=True)
      .over("category")                          # window: partition by category
      .alias("revenue_rank")
)

# 2. 7-day rolling average revenue per category
rolling = daily_cat_revenue.with_columns(
    pl.col("daily_revenue")
      .rolling_mean(window_size=7)               # sorted by order_date within group
      .over("category")
      .alias("rolling_7d_avg")
)

# 3. Month-over-month growth: aggregate to monthly first
monthly_cat = (
    enriched
    .group_by(["year_month", "category"])
    .agg(pl.col("net_amount").sum().alias("monthly_revenue"))
    .sort(["category", "year_month"])
    .with_columns(
        pl.col("monthly_revenue")
          .shift(1)                              # previous month's revenue
          .over("category")
          .alias("prev_month_revenue")
    )
    .with_columns(
        ((pl.col("monthly_revenue") - pl.col("prev_month_revenue"))
          / pl.col("prev_month_revenue") * 100)
        .round(2)
        .alias("mom_growth_pct")
    )
)

print("\nMonth-over-month growth by category (sample):")
print(monthly_cat.filter(pl.col("category") == "Electronics").head(6))

**What just happened?**
- `.over("category")` is Polars' window function syntax — equivalent to SQL `OVER (PARTITION BY category ORDER BY order_date)`.
- `rolling_mean(window_size=7)` computes a 7-day moving average within each category partition.
- **`shift(1).over("category")`** is the idiomatic Polars way to get the previous row's value within a group — used here for MoM growth.
- **No UDFs used anywhere** — all window operations run in Rust with full parallelism.

---
## Step 7 · Monthly summary aggregation

The final analytical output: monthly revenue by product category and region.

In [ ]:
monthly_summary = (
    enriched
    .group_by(["year_month", "category", "region"])
    .agg([
        pl.col("order_id").count().alias("num_orders"),
        pl.col("net_amount").sum().round(2).alias("total_revenue"),
        pl.col("net_amount").mean().round(2).alias("avg_order_value"),
        pl.col("customer_id").n_unique().alias("unique_customers"),
        pl.col("is_premium").mean().round(4).alias("premium_rate"),
    ])
    .sort(["year_month", "category", "region"])
)

print(f"Summary shape: {monthly_summary.shape}")
print(monthly_summary.head(10))

# Top 5 category-region combinations by total revenue
print("\nTop 5 by total revenue:")
print(
    monthly_summary
    .group_by(["category", "region"])
    .agg(pl.col("total_revenue").sum())
    .sort("total_revenue", descending=True)
    .head(5)
)

**What just happened?**
- A single `group_by().agg()` computes 5 metrics simultaneously — Polars evaluates all aggregations in a single pass over the data.
- `n_unique()` for counting distinct customers avoids a separate dedup step.
- `is_premium.mean()` computes the premium customer rate directly (True=1, False=0 in Boolean arithmetic).
- **This is the output your stakeholders would see** — the whole pipeline from raw CSV to this summary ran in under a second.

---
## Step 8 · Export: partitioned Parquet + summary CSV

In [ ]:
output_dir = os.path.join(TMPDIR, "output")
pathlib.Path(output_dir).mkdir(exist_ok=True)

# Export enriched orders partitioned by year_month
partition_dir = os.path.join(output_dir, "enriched_orders")
pathlib.Path(partition_dir).mkdir(exist_ok=True)

for (ym,), partition_df in enriched.group_by(["year_month"]):
    ym_dir = os.path.join(partition_dir, f"year_month={ym}")
    pathlib.Path(ym_dir).mkdir(exist_ok=True)
    partition_df.drop("year_month").write_parquet(
        os.path.join(ym_dir, "data.parquet"),
        compression="zstd",
    )

partitions = list(pathlib.Path(partition_dir).glob("year_month=*"))
print(f"Written {len(partitions)} monthly Parquet partitions")
print("Partitions:", sorted(p.name for p in partitions)[:5], "...")

# Export monthly summary to CSV
summary_path = os.path.join(output_dir, "monthly_summary.csv")
monthly_summary.write_csv(summary_path)
print(f"\nSummary CSV: {os.path.getsize(summary_path) / 1024:.1f} KB")

# Verify round-trip: scan one partition back
sample_ym = sorted(p.name for p in partitions)[0]
verify = pl.scan_parquet(
    os.path.join(partition_dir, sample_ym, "data.parquet")
).collect()
print(f"\nVerification read ({sample_ym}): {verify.shape[0]:,} rows")

**What just happened?**
- The enriched orders are exported as **Hive-style partitioned Parquet** — one directory per month.
- The monthly summary goes to CSV — small enough that CSV is fine and stakeholders can open it in Excel.
- **Round-trip verification** confirms the Parquet files are readable and structurally correct before declaring success.
- In production, replace the manual partition loop with a managed tool (dlt, dbt, or a cloud warehouse loader) for atomicity guarantees.

---
## Step 9 · Unit tests with pytest

Each transformation function is tested in isolation.
Good tests use minimal DataFrames — just enough rows to cover the edge cases.

In [ ]:
# Write tests to a file so pytest can discover and run them
test_code = '''
import polars as pl
import pytest

# --- functions under test (normally imported from your pipeline module) ---

def clean_orders(lf: pl.LazyFrame) -> pl.LazyFrame:
    return (
        lf
        .unique(subset=["order_id"], keep="first")
        .with_columns(pl.col("order_date").str.to_date("%Y-%m-%d"))
        .with_columns(pl.col("discount").fill_null(0.0))
        .filter(pl.col("status") != "cancelled")
    )

def engineer_order_features(lf: pl.LazyFrame) -> pl.LazyFrame:
    return lf.with_columns([
        pl.col("order_date").dt.year().alias("year"),
        pl.col("order_date").dt.month().alias("month"),
        pl.col("order_date").dt.strftime("%Y-%m").alias("year_month"),
    ])

# --- tests ---

def make_raw_orders():
    """Minimal fixture: 4 orders including 1 duplicate and 1 cancelled."""
    return pl.DataFrame({
        "order_id":   [1, 2, 2, 3],         # order_id=2 is duplicated
        "customer_id":[10, 20, 20, 30],
        "product_id": [100, 200, 200, 300],
        "quantity":   [1, 2, 2, 1],
        "discount":   [0.1, None, None, 0.0],  # None should become 0.0
        "status":     ["completed", "pending", "pending", "cancelled"],
        "order_date": ["2024-01-15", "2024-02-10", "2024-02-10", "2024-03-01"],
    }).lazy()

def test_clean_orders_deduplication():
    """Duplicate order_ids must be removed."""
    result = clean_orders(make_raw_orders()).collect()
    assert result["order_id"].n_unique() == result.shape[0], \
        "Duplicate order_ids remain after cleaning"

def test_clean_orders_removes_cancelled():
    """Cancelled orders must be excluded."""
    result = clean_orders(make_raw_orders()).collect()
    assert "cancelled" not in result["status"].to_list(), \
        "Cancelled orders found in cleaned output"

def test_clean_orders_fills_null_discount():
    """Null discounts must be filled with 0.0."""
    result = clean_orders(make_raw_orders()).collect()
    assert result["discount"].null_count() == 0, \
        "Null values remain in discount column"

def test_engineer_features_year_month():
    """year_month column must match the expected format."""
    raw = pl.DataFrame({
        "order_id": [1],
        "order_date": ["2024-06-15"],
        "discount": [0.0],
        "status": ["completed"],
    }).lazy()
    raw = raw.with_columns(pl.col("order_date").str.to_date("%Y-%m-%d"))
    result = engineer_order_features(raw).collect()
    assert result["year_month"][0] == "2024-06"
    assert result["year"][0] == 2024
    assert result["month"][0] == 6

def test_engineer_features_correct_row_count():
    """Feature engineering must not add or drop rows."""
    raw = make_raw_orders()
    cleaned = clean_orders(raw)
    featured = engineer_order_features(cleaned)
    n_clean = cleaned.collect().shape[0]
    n_feat  = featured.collect().shape[0]
    assert n_clean == n_feat, "Row count changed during feature engineering"
'''

import os
test_path = os.path.join(TMPDIR, "test_pipeline.py")
with open(test_path, "w") as f:
    f.write(test_code)

print(f"Tests written to: {test_path}")

In [ ]:
# Run the tests with pytest
import subprocess, sys

result = subprocess.run(
    [sys.executable, "-m", "pytest", test_path, "-v", "--tb=short"],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

**What just happened?**
- Each test function targets exactly **one behaviour** of one transformation function.
- Tests use a **minimal fixture** (4 rows) that covers all edge cases — not the full 50k-row dataset.
- **Fast, isolated tests** are the key: these 5 tests run in under 1 second and can run in CI without any external dependencies.
- The pattern: `make_*()` returns the minimal input, the test calls the transformation, and asserts on the output.

---
## Production readiness checklist

Before shipping a Polars pipeline to production, verify each item:

| # | Checklist item | Why it matters |
|---|---|---|
| 1 | **All ingestion uses `scan_*` (lazy)** | Avoids loading more data than needed |
| 2 | **Schema validated at ingest** | Catches bad files before any compute |
| 3 | **No UDFs in the hot path** | UDFs kill parallelism and optimizer visibility |
| 4 | **`rechunk()` after many concat ops** | Prevents memory fragmentation slowdowns |
| 5 | **Categorical dtypes for low-cardinality strings** | 10–50x memory reduction |
| 6 | **Output partitioned by the most common filter column** | Enables partition pruning at read time |
| 7 | **Unit tests for each transformation function** | Catch regressions in CI before they reach production |
| 8 | **`.explain(optimized=True)` reviewed before ship** | Confirm pushdowns are applied |
| 9 | **`streaming=True` for pipelines on large data** | Bounded memory, works beyond RAM |
| 10 | **Benchmarked on representative data volume** | Performance on 1k rows does not predict 100M rows |

---
## Portfolio README template

Copy this to your GitHub repo's `README.md` — it frames the work for employers and peers.

```markdown
# Polars E-Commerce Analytics Pipeline

A production-quality analytics pipeline built with **Polars** as a capstone
for the [certified-journeys / polars-certified](https://certified-journeys.github.io/courses/polars-certified/) course.

## What it does
- Ingests 3 related CSV tables (~50k rows each) with schema validation
- Cleans: null handling, type casting, deduplication (pure Polars expressions)
- Feature engineering: date parts, rolling windows, MoM growth (no UDFs)
- Multi-table join pipeline producing enriched orders
- Window analytics: revenue rank per category, 7-day rolling revenue
- Aggregation: monthly summary by product category and region
- Export: Parquet (partitioned by month) + summary CSV
- Unit-tested with pytest (5 tests, < 1s total runtime)

## Tech stack
- Python 3.10+
- [Polars](https://pola.rs) 0.20+ for all data processing
- [PyArrow](https://arrow.apache.org/docs/python/) for Parquet I/O
- pytest for unit tests

## Run it
```bash
pip install polars pyarrow pytest
jupyter notebook day-10-capstone-production-analytics-pipeline.ipynb
```

## Results
<!-- Add a screenshot of the monthly_summary output here -->

## What I learned
<!-- 3-5 sentences on what was most surprising or useful -->
```

---
## Day 10 recap

| Stage | Polars pattern used |
|---|---|
| Synthetic data generation | `pl.DataFrame({...})` with numpy |
| Lazy ingestion + schema validation | `scan_csv` + `schema_overrides` + `collect_schema()` |
| Cleaning | Composable `LazyFrame → LazyFrame` functions |
| Feature engineering | Pure expressions: `dt.year()`, `dt.strftime()` |
| Multi-table join | Chained `.join()` with explicit `.select()` before join |
| Window analytics | `.over()`, `rolling_mean()`, `shift()` |
| Aggregation | `group_by().agg()` with 5 metrics in one pass |
| Export | Partitioned Parquet (by month) + summary CSV |
| Unit tests | pytest with minimal fixtures, one behaviour per test |

> **Tip:** A well-written Polars pipeline on GitHub demonstrates more to employers than any course certificate. The pattern in this notebook — composable lazy transformations, schema validation at ingest, unit-tested functions, partitioned Parquet output — is exactly what senior data engineers look for in a portfolio project.

---
## Congratulations! 🎉

You have completed **polars-certified** — 10 days covering:

- Days 1–3: DataFrames, expressions, lazy execution
- Days 4–5: String, datetime, and list operations
- Day 6: Joins, concatenation, and reshaping
- Day 7: Parquet, CSV, JSON, and DuckDB I/O
- Day 8: Performance profiling and optimization
- Day 9: Custom functions and UDFs
- **Day 10: Production pipeline capstone** ← you are here

### Publish your work

1. Fork this notebook to your own GitHub repo
2. Copy the README template above and fill in your results
3. Share the link — tag `#polars` and `#certifiedjourneys`

### What's next?

The natural next step is **[dlt-certified](https://certified-journeys.github.io/courses/dlt-certified/)** — the data engineering companion to this course.

**dlt** (data load tool) solves the problem of *getting data into* your Polars pipelines at production scale: incremental loads, schema evolution, secrets management, and connectors to 100+ sources. Together, dlt + Polars is a modern, fully-Python data engineering stack that replaces Spark + Airflow for most analytical workloads.

| You learned | dlt-certified adds |
|---|---|
| Transform data with Polars | Ingest data from APIs, databases, files |
| Write Parquet output | Manage schema evolution and incremental state |
| Unit-test transformations | Deploy pipelines to DuckDB, BigQuery, Snowflake |
| Partition by date | Automatic incremental + merge strategies |

**[Start dlt-certified →](https://certified-journeys.github.io/courses/dlt-certified/)**

---
Track your completed journey at [certified-journeys.github.io](https://certified-journeys.github.io).